# Probe and flash — SWD, RTT and DFU

A simulated STM32F411 (`VirtualMcu`) sits behind a virtual CMSIS-DAP probe. We read its memory, flash new firmware, read its RTT log and send it a command, then update it again through its USB DFU bootloader. With a real board use `DebugProbe.Create(o => o.UseTarget("STM32F411CEUx"))` (probe-rs) and `DfuClient.Create(o => o.UseDevice(0x0483, 0xDF11))`.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.30.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.30.0-preview.1"

## Firmware images
Intel HEX, DFU files and ELF load into one model: segments, an entry point, CRCs.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Debug;
using IoTCom.Net.Transport.Usb;

var image = VirtualMcu.BuildDemoFirmware("1.1.0");
Console.WriteLine($"{image}, version tag {image.FindTag("FWVER:")}");
var hex = IntelHex.Format(image);
Console.WriteLine(string.Join("\n", hex.Split('\n').Take(3)));
var dfuFile = DfuFile.Parse(DfuFile.BuildDfuSe(image, 0x0483, 0xDF11));
Console.WriteLine(dfuFile.Describe());

## Read-only probe
Memory reads and the core state work while the firmware runs; writes are refused.

In [ ]:
var mcu = new VirtualMcu(firmware: VirtualMcu.BuildDemoFirmware("1.0.0"));
mcu.Step(1.5);
var viewer = DebugProbe.Create(o => o.UseVirtual(new VirtualProbeBackend(mcu)).UseTarget("STM32F411CEUx"));
Console.WriteLine($"core {await viewer.CoreAsync()}, uptime {await viewer.ReadUInt32Async(VirtualMcu.UptimeAddress)} ms");
var rtt = await viewer.AttachRttAsync(VirtualMcu.RamBase, VirtualMcu.RamSize);
Console.Write(await rtt.ReadTextAsync());
try { await viewer.FlashAsync(image); } catch (ReadOnlyModeException e) { Console.WriteLine(e.Message); }
await viewer.DisposeAsync();

## Flash over SWD, talk over RTT
Erase, program, verify, reset — then the new firmware boots and answers on the down channel.

In [ ]:
var engineer = DebugProbe.Create(o => o.UseVirtual(new VirtualProbeBackend(mcu)).UseTarget("STM32F411CEUx").AllowWrites());
await engineer.FlashAsync(image, new Progress<FlashProgress>(p => { if (p.Fraction >= 1) Console.WriteLine($"{p.Phase} done"); }));
mcu.Step(0.5);
var terminal = await engineer.AttachRttAsync(VirtualMcu.RamBase, VirtualMcu.RamSize);
Console.Write(await terminal.ReadTextAsync());
await terminal.WriteAsync(System.Text.Encoding.ASCII.GetBytes("led on\n"));
mcu.Step(0.1);
Console.Write(await terminal.ReadTextAsync());
Console.WriteLine($"LED {(mcu.Led ? "on" : "off")}");
await engineer.DisposeAsync();

## Update over USB DFU
The same chip in its DfuSe bootloader: the memory layout comes from the interface string; page erase, set address, blocks, leave.

In [ ]:
var bus = new VirtualUsbBus();
var bootloader = bus.Add(new VirtualDfuDevice(mcu));
var dfu = DfuClient.Create(o => o.UseVirtual(bus).AllowWrites());
await dfu.ConnectAsync();
Console.WriteLine(dfu.Layout!.Describe());
await dfu.DownloadAsync(VirtualMcu.BuildDemoFirmware("2.0.0"));
await dfu.LeaveAsync();
Console.WriteLine($"{bootloader.PagesErased} pages erased, {bootloader.BlocksWritten} blocks; now running {mcu.FirmwareVersion}");
await dfu.DisposeAsync();

## Going further
`iotcom probe list`, `iotcom probe flash <file> --target <chip> --allow-write`, `iotcom probe rtt --sim`, `iotcom dfu download --sim --leave --allow-write`, the Gallery's *Probe and flash firmware* and the ProbeFlash sample. See `docs/en/protocols/debug.md`.

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*